<a href="https://colab.research.google.com/github/sanaisrail/urdu-ocr-codesaviours-si26-Sana/blob/main/week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch pillow pandas

In [3]:
!pip install sentencepiece

In [5]:
!pip uninstall -y tokenizers transformers sentencepiece
!pip install transformers==4.38.2 tokenizers==0.15.2 sentencepiece protobuf

Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: sentencepiece 0.2.2
Uninstalling sentencepiece-0.2.2:
  Successfully uninstalled sentencepiece-0.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
ERROR: pip's dependency resolver

In [1]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

print("Processor loaded")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Processor loaded


In [7]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd


class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        image = Image.open(row["image"]).convert("RGB")

        encoding = self.processor(
            image,
            return_tensors="pt"
        )

        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [10]:
import os

print(os.listdir())

['.config', 'sample_data']


In [11]:
from google.colab import files
uploaded = files.upload()

Saving labels (23).csv to labels (23).csv


In [14]:
dataset = UrduOCRDataset(
    "labels (23).csv",
    processor
)

Dataset loaded: 200 samples


In [16]:
sample = dataset[0]

print("Pixel values shape:", sample["pixel_values"].shape)
print("Labels shape:", sample["labels"].shape)
print("Dataset is working correctly!")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/book/1.png'